# Sheet 02 - Multilayer Perceptrons

Introduction to Deep Learning - Summer Semester 2026

Ulf Krumnack & Robin Rawiel - Universität OsnabrückDue: April 26, 2026

## Task 1: Theory Questions \[6 points\]

### 1.1 Activations & Nonlinearity \[1 point\]

1.  Write down the formulas for the **sigmoid** $\sigma(z)$ and **ReLU**
    activation functions. What is the output range of each?

<span style="color: green;"><strong>Answer (1.1.1).</strong></span> The **sigmoid** (logistic) function is  
$$\sigma(z) = \frac{1}{1 + e^{-z}}.$$  
Its **output range** is the open interval **$(0, 1)$** (the endpoints are approached as $z \to \mp\infty$ but are never exactly reached for finite $z$).  
The **ReLU** (rectified linear unit) is  
$$\mathrm{ReLU}(z) = \max(0, z) =
    \begin{cases} z & \text{if } z > 0, \\ 0 & \text{if } z \le 0. \end{cases}$$  
Its **output range** is the half-line **$[0, \infty)$** (all nonnegative real numbers, including 0 at $z \le 0$).  

1.  Can a neural network with only linear activation functions learn
    non-linear decision boundaries? Explain why or why not.

<span style="color: green;"><strong>Answer (1.1.2).</strong></span> **No.** Linear layers with linear (e.g. identity) activations always compose to **one** affine map $\hat{y} = W x + b$, so the model is still linear in $x$ and the decision boundary is a **hyperplane**. **Nonlinear** boundaries need **nonlinear** activations. 

### 1.2 Computational Graphs & Backpropagation \[2 points\]

1.  *(1 point)* Draw the **computational graph** for a one-hidden-layer
    MLP ($x \to z^{(1)} \to a^{(1)} \to z^{(2)} \to \hat{y}$) in
    explicit style, where you show both value nodes and operation nodes.
    Label the dimensions of the weight matrices and activation vectors
    for an input of size $d$, a hidden layer of size $h$, and a single
    output.

<details><summary>Additional section</summary>

1. Explicit style: both value nodes and operation nodes (a bipartite graph). This is the most explicit representation: rectangles for values, circles for operations. An operation node takes incoming arrows from its input values and has an outgoing arrow to its output value.  
2. In explicit style, we can annotate the dimension next to the value node, for example "x ∈ ℝ^d".

</details>

<span style="color: green;"><strong>Answer (1.2.1).</strong></span> One-hidden-layer MLP in explicit style (bipartite graph: value rectangles, operation circles; dimension labels on value nodes as in the figure).  

![Computational graph for one hidden-layer MLP](./1.2.1%20Computational%20Graph.png)

1.  *(1 point)* In the backward pass, the error signal $\delta^{(\ell)}$
    is propagated from layer $\ell$ to layer $\ell-1$ via the recursion
    $$\delta^{(\ell-1)} = \bigl(W^{(\ell)}\bigr)^\top \delta^{(\ell)} \;\odot\; \varphi'\bigl(z^{(\ell-1)}\bigr)$$
    and the weight gradient is computed as
    $$\frac{\partial \mathcal{L}}{\partial W^{(\ell)}} = \delta^{(\ell)} \bigl(a^{(\ell-1)}\bigr)^\top.$$
    Explain **in words** the role of each component: (a) the transpose
    $(W^{(\ell)})^\top$, (b) the element-wise product
    $\odot\, \varphi'(z^{(\ell-1)})$, and (c) the outer product
    $\delta^{(\ell)} (a^{(\ell-1)})^\top$.

<span style="color: green;"><strong>Answer (1.2.2).</strong></span>  

**Definitions (symbols in the formulas).**  
$z^{(\ell)}$ = **pre-activations** in layer $\ell$;  
$a^{(\ell)} = \varphi\!\left(z^{(\ell)}\right)$ = **activations** (element-wise nonlinearity $\varphi$).  
$W^{(\ell)}$ = **weight matrix** in the map $a^{(\ell-1)} \mapsto W^{(\ell)} a^{(\ell-1)} + b^{(\ell)}$.  
$\delta^{(\ell)} = \dfrac{\partial \mathcal{L}}{\partial z^{(\ell)}}$ = **error signal** (sensitivity of the loss w.r.t. $z^{(\ell)}$).  
$\varphi'\!\left(z^{(\ell-1)}\right)$ = **element-wise** derivative: $\varphi'\!\left(z^{(\ell-1)}\right)_i = \varphi'\!\left(z_i^{(\ell-1)}\right)$.  
$\odot$ = **element-wise (Hadamard) product** of two vectors of the same length.  
For column vectors $u\in\mathbb{R}^{m}$, $v\in\mathbb{R}^{n}$:  
$u v^\top$ = **outer product** (an $m\times n$ matrix with entries $u_i v_k$).  

**(a) $(W^{(\ell)})^\top$** — Linearity of the forward step $z^{(\ell)} = W^{(\ell)} a^{(\ell-1)}+\cdots$: backward pass multiplies the upstream vector $\delta^{(\ell)}$ by $W^\top$ to get $\partial\mathcal{L} / \partial a^{(\ell-1)}$ (before applying $\varphi'$).  

**(b) $\odot\,\varphi'\!\left(z^{(\ell-1)}\right)$** — Chain rule for $a^{(\ell-1)} = \varphi\!\left(z^{(\ell-1)}\right)$: multiply $\partial\mathcal{L} / \partial a^{(\ell-1)}$ by the local slopes $\varphi'\!\left(z^{(\ell-1)}\right)$ to obtain $\delta^{(\ell-1)} = \partial\mathcal{L} / \partial z^{(\ell-1)}$.  

**(c) $\delta^{(\ell)}(a^{(\ell-1)})^\top$** — For $z_i^{(\ell)} = \sum_k W^{(\ell)}_{ik} a_k^{(\ell-1)} + \cdots$ we have $\dfrac{\partial z_i^{(\ell)}}{\partial W^{(\ell)}_{ik}} = a_k^{(\ell-1)}$, hence $\dfrac{\partial\mathcal{L}}{\partial W^{(\ell)}_{ik}} = \delta_i^{(\ell)} a_k^{(\ell-1)}$, i.e. the matrix is the **outer product** of $\delta^{(\ell)}$ and $a^{(\ell-1)}$.  

### 1.3 Regularization \[2 points\]

1.  *(0.5 points)* How does **early stopping** work? What pattern in the
    learning curves (training loss vs. validation loss) tells you when
    to stop?

<span style="color: green;"><strong>Answer (1.3.1).</strong></span>  
**What it is:** During training, you also evaluate the model from time to time on a **validation set** (data not used for the gradient). **Early stopping** means you **do not** train as long as the training loss alone would suggest, but you **stop** (or at least *consider* stopping) when the model’s **validation** performance is no longer getting better, and you keep the parameters that achieved the **lowest** validation error so far.  

**Lecture figure (error vs. model complexity; training vs. validation or test):**  

<img src="./Tranning%20loss%20vs%20validation%20loss.png" alt="Training vs validation (test) error vs. model complexity" height="450"/>

The **blue** curve = training error (keeps going **down** as the model can fit the training set better). The **red** curve = **validation** / test error: it has a **minimum** at **“good complexity”** (best generalization), then **rises** in the **overfitting** region. **Underfitting** is on the left, where *both* errors are high.  

**Over epochs** you see the *same* pattern in terms of *effective* behavior:  
- For a long time, both **training** and **validation** loss often **decrease** together.  
- Later, **training** loss can keep **falling** while **validation** loss **levels off** and then **increases** (the growing **gap** = **overfitting**).  

**When to early-stop:** not at the best training loss, but around the **lowest validation loss** (or when validation **consistently worsens**) — the analogue of the **red minimum** in the picture.  


1.  *(0.5 points)* How does **dropout** work during training vs. during
    inference?

<span style="color: green;"><strong>Answer (1.3.2).</strong></span>  
**During training:** At each forward pass, you **randomly** set a fraction of activations to **zero** (according to a fixed probability, e.g. a Bernoulli mask per unit). The network therefore behaves like a **smaller, different** sub-network on each pass, which **reduces** co-adaptation of features and has a **regularizing** effect.  

**During inference (testing / evaluation):** You **turn off** random dropping: all units are **active**. The raw outputs would then be **larger in expectation** than in training, so you **re-scale** (typically multiply by the **keep-probability** $1-p$ for dropped units) so that the **expected** value of each activation matches what training saw.  
In a common variant **(inverted dropout)**, the compensating **scaling** is done **during** training (divide by keep-probability) so that **inference** can use the network **without** extra scaling.  

1.  *(1 point)* Write the **L2-regularized loss** function. How does the
    regularization parameter $\lambda$ affect the weights during
    training?

<span style="color: green;"><strong>Answer (1.3.3).</strong></span>  
**L2** regularization (same *spirit* as **ridge** regression in the lecture) adds a **penalty** that grows with the **squared** magnitude of the weights, so the total cost is:  

$$\mathcal{J} = \mathcal{L} + \frac{\lambda}{2} \sum_{\ell} \bigl\|W^{(\ell)}\bigr\|_F^2,$$  

where **$\mathcal{L}$** is the usual data loss (e.g. MSE, cross-entropy) and  
**$\|W^{(\ell)}\|_F^2$** = sum of the **squares of all entries** in the weight matrix $W^{(\ell)}$ (Frobenius norm squared; your course may or may not use the **$1/2$** factor — it is only a convention so the gradient looks simple).  

**Effect of $\lambda$:**  
- The gradient of the penalty term is **proportional to** $W^{(\ell)}$, so each update is pulled toward **smaller** norms.  
- **Larger** $\lambda$ = **stronger** pull → weights stay **closer to zero** and the fit is often **smoother** and less able to “chase” noise (compare **ridge** vs. **unregularized** fit in the small-noisy-dataset example in the lecture).  
- **$\lambda = 0$** removes the penalty: you are back to minimizing **$\mathcal{L}$** only.  


### 1.4 Mini-batch Learning \[1 point\]

1.  *(0.5 points)* Explain the difference between **batch gradient
    descent**, **mini-batch gradient descent**, and **stochastic
    gradient descent (SGD)**.

<span style="color: green;"><strong>Answer (1.4.1).</strong></span>  

**Terminology:**  
- **Example** (also **training example** / **data point**): one item from the training set. For **supervised** learning it is a pair **$(x, y)$** — an **input** (feature vector) and, if applicable, a **label** / target. *“$N$ examples”* = $N$ such pairs.  
- **Parameter:** a **learnable** quantity in the model that is adjusted during training, e.g. **entries of weight matrices and bias vectors** in a neural network. (Not the hyperparameters you set by hand, like learning rate or batch size.)  
- **Step** (an **update step** / **optimizer step**): one cycle of *compute the loss on the chosen data subset → compute the gradient w.r.t. the **parameters** → **change** the parameters* (e.g. $\theta \leftarrow \theta - \eta\, g$ in gradient descent, with step size/learning rate $\eta$).  

**Batch gradient descent (full batch):** one parameter update per step, using the loss gradient
$\nabla\mathcal{L} = \frac{1}{N}\sum_{i=1}^{N} \nabla \ell_i$ computed over the **entire** training set of $N$ examples (or equivalently, the unscaled sum, depending on convention).  

**Mini-batch gradient descent:** the training set is split into **mini-batches** of $B$ examples ($1 < B < N$). One update uses the gradient **averaged (or summed) only over the current batch** of $B$ points, then the next batch, until all data are used.  

**Stochastic gradient descent (SGD, online / pure SGD in the strict sense):** each update uses the gradient of the loss for **a single** example, i.e. **$B=1$** (one random example per step, often after shuffling the data each **epoch**).  

**Comparison (typical $N$ training examples, batch size $B$):**  

| **Criterion** | **Batch (full)** | **Mini-batch** | **SGD (strict)**  
|---|---|---|---|  
| **Examples per update** | $N$ (all) | $B$ with $1<B<N$ | $1$  
| **# updates per epoch** | $1$ | $\lceil N/B \rceil$ (if batches partition the set) | $N$  
| **Cost of one update** | high (one pass over all data) | medium | low  
| **Noise in the gradient** | none (empirical full-batch gradient) | small–moderate (depends on $B$) | large  
| **Idea** | exact steepest step for the *batch* training loss (expensive) | trade-off: cheaper updates + noise can help | very cheap, noisy; often with small learning rate or averaging  

1.  *(0.5 points)* Define the term **epoch**. If you have 10,000
    training examples and use a mini-batch size of 250, how many
    parameter updates are performed per epoch?

<span style="color: green;"><strong>Answer (1.4.2).</strong></span> An **epoch** is one **full pass** over the training set: every training example is used (typically once), often split into **mini-batches** in a fixed or shuffled order.  

With $N=10{,}000$ and mini-batch size $B=250$, the number of batches (and, with **one** parameter update per batch, the number of updates per epoch) is  

$$10{,}000 / 250 = 40.$$  

So there are **40** weight updates per epoch.  

## Task 2: MLP from Scratch \[8 points\]

**Learning objectives:**

- Implement forward and backward pass manually in NumPy
- Understand backpropagation mechanics
- Optional: verify correctness via gradient checking

### 2.1 Activation Functions \[1 point\]

Implement these **element-wise** for NumPy arrays (same shape in / out).  

**ReLU** (rectified linear unit)  
$$\mathrm{ReLU}(z) = \max(0, z).$$  
Derivative (subgradient; at $z=0$ use **$0$** in code, as usual):  
$$\frac{d}{dz}\mathrm{ReLU}(z) = \begin{cases} 1 & \text{if } z > 0, \\ 0 & \text{if } z \le 0. \end{cases}$$  

**Sigmoid** (logistic)  
$$\sigma(z) = \frac{1}{1 + e^{-z}}.$$  
Derivative:  
$$\sigma'(z) = \sigma(z)\,\bigl(1 - \sigma(z)\bigr).$$  


In [14]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

np.random.seed(42)


def relu(z):
    return np.maximum(0, z)


def relu_derivative(z):
    return np.where(z > 0, 1.0, 0.0)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def sigmoid_derivative(z):
    a = sigmoid(z)
    return a * (1 - a)

In [15]:
# 2.1 — Tests (run the activations cell above first)

# ReLU
z = np.array([[-1.0, 0.0, 0.5, 2.0], [-2.0, 3.0, -0.0, 1.0]])
exp_relu = np.array([[0.0, 0.0, 0.5, 2.0], [0.0, 3.0, 0.0, 1.0]])
assert np.allclose(relu(z), exp_relu)
assert relu(5.0) == 5.0
assert relu(-3.0) == 0.0

z = np.array([-1.0, 0.0, 0.0, 1.0, 2.0])
exp_relu_d = np.array([0.0, 0.0, 0.0, 1.0, 1.0])
assert np.allclose(relu_derivative(z), exp_relu_d)

# Sigmoid
assert np.isclose(sigmoid(0.0), 0.5)
z = np.array([[-6.0, 0.0, 6.0]])
exp_sig = np.array([[0.0024726231566347743, 0.5, 0.9975273768433653]])
assert np.allclose(sigmoid(z), exp_sig, rtol=1e-7, atol=1e-7)

z = np.array([-1.0, 0.0, 1.0])
exp_sig_d = np.array([0.19661193324148185, 0.25, 0.19661193324148185])
assert np.allclose(sigmoid_derivative(z), exp_sig_d, rtol=1e-7, atol=1e-7)

print("all 2.1 tests passed")


all 2.1 tests passed


### 2.2 MLP Class \[3 points\]

Implement a multi-layer perceptron with configurable layer sizes, small
random weight initialization, ReLU hidden activations, and sigmoid
output.

In [ ]:
class MLP:
    def __init__(self, layer_sizes):
        """Initialize MLP with given layer sizes, e.g. [2, 32, 16, 1].

        Use small random initialization (scale 0.1) for weights and zeros for biases.
        """
        # TODO: Your solution here

    def forward(self, X):
        """Forward pass. Store intermediate values for backprop.

        Args:
            X: Input, shape (N, D).

        Returns:
            Output, shape (N, 1).
        """
        # TODO: Your solution here

    def compute_loss(self, y_true, y_pred):
        """Binary cross-entropy loss."""
        # TODO: Your solution here

    def backward(self, y_true):
        """Backward pass. Compute gradients for all weights and biases.

        Args:
            y_true: True labels, shape (N, 1).

        Returns:
            dW: List of weight gradients.
            db: List of bias gradients.
        """
        # TODO: Your solution here

    def train_step(self, X, y, lr=0.1):
        """One forward + backward pass with parameter update."""
        # TODO: Your solution here

### 2.3 Train on Moons Dataset \[2 points\]

In [ ]:
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
y_train = y_train.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

In [ ]:
# TODO: Your solution here

### 2.4 Visualize Decision Boundary \[1 point\]

In [ ]:
def plot_decision_boundary(model, X, y):
    """Plot the decision boundary of the MLP."""
    # TODO: Your solution here

In [ ]:
# TODO: Your solution here

### 2.5 Gradient Checking (Bonus) \[1 point\]

Verify your backpropagation implementation by comparing analytical
gradients with numerical approximations using $\epsilon = 10^{-7}$.

In [ ]:
def gradient_check(model, X, y, epsilon=1e-7):
    """Compare analytical and numerical gradients."""
    # TODO: Your solution here

In [ ]:
# TODO: Your solution here

## Task 3: MLP with PyTorch \[6 points\]

**Learning objectives:**

- Learn the PyTorch API (`nn.Module`, `optim`, `DataLoader`)
- Compare different regularization methods
- Understand the effect of network depth and width

### 3.1 Setup and Model \[2 points\]

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_t = torch.FloatTensor(X_train).to(device)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1).to(device)
X_test_t = torch.FloatTensor(X_test).to(device)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1).to(device)
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim=2, hidden_dims=[128, 128, 128], output_dim=1, dropout=0.0):
        """MLP with configurable hidden layers and optional dropout.

        Args:
            input_dim: Number of input features.
            hidden_dims: List of hidden layer sizes.
            output_dim: Number of outputs.
            dropout: Dropout probability (0 = no dropout).
        """
        super().__init__()
        # TODO: Your solution here

    def forward(self, x):
        # TODO: Your solution here

### 3.2 Training Function \[2 points\]

In [ ]:
def train_model(model, train_loader, X_test_t, y_test_t, lr=0.001, weight_decay=0.0, epochs=200):
    """Train the model and track metrics.

    Args:
        model: PyTorch model.
        train_loader: DataLoader for training data.
        X_test_t, y_test_t: Test tensors.
        lr: Learning rate.
        weight_decay: L2 regularization (passed to optimizer).
        epochs: Number of epochs.

    Returns:
        Dictionary with train_losses, test_losses, train_accs, test_accs.
    """
    # TODO: Your solution here

### 3.3 Regularization Comparison \[1 point\]

Train five configurations and compare:

1.  **Baseline:** `[128, 128, 128]`, no regularization
2.  **L2:** weight_decay = 0.01
3.  **Dropout:** dropout = 0.3
4.  **L2 + Dropout:** both
5.  **Small network:** `[16]`

In [ ]:
# TODO: Your solution here

### 3.4 Depth vs. Width \[1 point\]

Compare three architectures with roughly the same total parameters:

- **Wide & Shallow:** `[256]`
- **Medium:** `[64, 64]`
- **Deep & Narrow:** `[16, 16, 16, 16, 16]`

In [ ]:
# TODO: Your solution here

### 3.5 Reflection

1.  How does the PyTorch implementation compare to your NumPy version in
    terms of code complexity and training speed?
2.  Which regularization approach worked best? Why?
3.  What tradeoffs do you observe between deep-narrow and wide-shallow
    architectures?